# 지우 final CN7 실행 결과 확인
원본 CLI는 Python3.11 별도환경에서 실행. 이 노트북은 결과를 읽어 행/Shot 및 불량 유형별 지표를 정리한다.

In [1]:
from pathlib import Path
import pandas as pd,json
from sklearn.metrics import confusion_matrix,average_precision_score
root=Path.cwd()
if not (root/'reports').exists():root=root.parent
package=root/'tmp/jiwoo_final_6ca6729_c0859f2aa953421a859d31dbc31e5e81/JIWOO/final'
ledger=pd.read_csv(package/'outputs/ledger/ledger_v1.csv',parse_dates=['TimeStamp'])
cn=ledger[ledger['product'].eq('CN7')].copy();cn['Reason']=cn.Reason.fillna('정상')
rows=[]
for per,z in cn.groupby('period'):
 for unit,q in [('row',z),('Shot',z.groupby('TimeStamp',as_index=False).agg(y=('y','max'),alarm=('alarm','max'),score=('score','first')))]:
  tn,fp,fn,tp=confusion_matrix(q.y,q.alarm,labels=[0,1]).ravel()
  rows.append(dict(period=per,unit=unit,N=len(q),bad=int(q.y.sum()),alarms=int(q.alarm.sum()),TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),precision=tp/(tp+fp),recall=tp/(tp+fn),F1=2*tp/(2*tp+fp+fn),AP_score=average_precision_score(q.y,q.score)))
metrics=pd.DataFrame(rows)
reasons=cn[cn.y.eq(1)].groupby(['period','Reason']).agg(actual=('y','size'),found=('alarm','sum')).reset_index()
# Per-Shot reason combination shows whether gas TP is independent of shortshot.
types=cn[cn.y.eq(1)].groupby(['period','TimeStamp']).agg(reasons=('Reason',lambda s:'·'.join(sorted(set(s)))),alarm=('alarm','max')).reset_index()
selfcheck=pd.read_csv(package/'outputs/ledger/selfcheck.csv')
assert len(selfcheck)==28 and selfcheck['판정'].eq('PASS').all()
metrics.to_csv(root/'reports/tables/jiwoo_final_cn7_reproduced_metrics.csv',index=False,encoding='utf-8-sig')
reasons.to_csv(root/'reports/tables/jiwoo_final_cn7_defect_type_counts.csv',index=False,encoding='utf-8-sig')
report='# 지우 final CN7 실제 실행 결과\n\n'
report+='Source origin/jiwoo commit6ca6729. 원본 코드 수정·브랜치 병합 없이 별도 임시 폴더에서 python run_all.py --kappa10 실행. Python3.11.8, numpy2.4.4,pandas3.0.2,sklearn1.8.0,scipy1.17.1,joblib1.5.3 등 기본 실행 의존성을 원본 고정 버전으로 설치했다. 선택 --experiments 및 torch/TabPFN/Chronos 탐색은 실행하지 않았다. 기본1~9단계41초, 자체재현28/28PASS.\n\n'
report+='17개 인과 편차 피처의 Ledoit-Wolf Mahalanobis 탐지기는 비라벨 정상 기준으로 학습. 경보는 pct>=.999. 확률 층은별도로 라벨을 이용한 로지스틱/Venn-Abers/Beta 측면 보정이며 AP_score와 구분. 정상비라벨 가정·기준 선택은 모델 평가 한계에 남는다.\n\n'
report+=metrics.to_csv(index=False)+'\n## 유형별 행 탐지\n'+reasons.to_csv(index=False)
report+='\n## 해석\n공식CN7 미성형4행(3Shot) 전부 탐지. 가스13행 중2행만탐지했으며,두행은미성형과동시에기록된Shot의LH가스이다. 순수가스11Shot은전부놓침. 추가기간은초기허용불량10행(5Shot)만탐지하고 미성형1행은놓침. 공식/추가 전체불량성적이지가스전용성적이아니다.\n\n'
report+='추가기간Mahalanobis 설정/임계값은추가결과를본뒤선정된탐색평가임을원본README가명시한다. 자체PASS는숫자재현을뜻하며새자료독립성능인증아님. CN7/RG3 또는Shot/행과다른불량타깃성적을직접비교하지않는다.\n'
(root/'reports/jiwoo_final_cn7_reproduction.md').write_text(report,encoding='utf-8')
print(metrics.to_string(index=False))
print(reasons.to_string(index=False))
print(types.to_string(index=False))


  period unit    N  bad  alarms  TP  FP  FN   TN  precision   recall       F1  AP_score
   extra  row 2745   11      16  10   6   1 2728      0.625 0.909091 0.740741  0.797347
   extra Shot 1373    6       8   5   3   1 1364      0.625 0.833333 0.714286  0.731311
official  row 1211   17      10   6   4  11 1190      0.600 0.352941 0.444444  0.372833
official Shot  606   14       5   3   2  11  590      0.600 0.214286 0.315789  0.246891
  period Reason  actual  found
   extra    미성형       1      0
   extra 초기허용불량      10     10
official     가스      13      2
official    미성형       4      4
  period           TimeStamp reasons  alarm
   extra 2020-10-27 00:56:21  초기허용불량      1
   extra 2020-10-27 00:59:00  초기허용불량      1
   extra 2020-10-27 01:03:34  초기허용불량      1
   extra 2020-10-27 01:04:41  초기허용불량      1
   extra 2020-10-27 01:05:41  초기허용불량      1
   extra 2020-11-03 04:44:17     미성형      0
official 2020-10-16 05:21:38      가스      0
official 2020-10-16 05:37:29      가스      0
official 